# Self-Supervised Model Inspection

本笔记本用于自监督模型的推理与可视化：展示网络输出（重建/obj）、预测的 Zernike 系数、RMS、Zernike 相位图，以及对应的真实 obj 图像、真实 Zernike 相位图与系数。

参照：inspect_supervised_model.ipynb 的结构，必要时修改路径和数据字段以配合你的数据集/模型输出。

## 1. 环境与依赖
如果缺少依赖，请在环境中安装，例如：
```bash
pip install -e . matplotlib torch numpy
```

In [ ]:
from __future__ import annotations

import csv
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
from torch.utils.data import Dataset

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / 'src').exists():
    REPO_ROOT = Path.cwd().parent
sys.path.insert(0, str(REPO_ROOT / 'src'))

from ao2d.data import build_dataloader, get_data_root, resolve_path
from ao2d.data.io import load_image, normalize01
from ao2d.data.paths import infer_manifest_data_root, resolve_manifest_record_path
from ao2d.models.factory import make_model
from ao2d.optics import generate_psf2d_from_zernike, pupil_coordinates_2d, zernike_wavefront

plt.rcParams['figure.dpi'] = 120

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('repo:', REPO_ROOT)
print('device:', device)


def zernike_mode_count(zernike_indices=None, optics_cfg=None):
    if zernike_indices is not None:
        return len(tuple(zernike_indices))
    if optics_cfg and optics_cfg.get('zernike_indices') is not None:
        return len(optics_cfg['zernike_indices'])
    return 13


def as_zernike_coeffs(coeffs, zernike_indices=None, optics_cfg=None):
    if coeffs is None:
        return np.zeros(zernike_mode_count(zernike_indices, optics_cfg), dtype=np.float32)
    coeffs = np.asarray(coeffs, dtype=np.float32).squeeze()
    if coeffs.ndim == 0:
        coeffs = coeffs[None]
    target_len = zernike_mode_count(zernike_indices, optics_cfg)
    if coeffs.size == target_len:
        return coeffs
    if coeffs.size > target_len:
        return coeffs[:target_len]
    padded = np.zeros(target_len, dtype=np.float32)
    padded[: coeffs.size] = coeffs
    return padded


def coeffs_to_wavefront(
    coeffs,
    image_size,
    zernike_indices=None,
    optics_cfg=None,
    device='cpu',
    dtype=torch.float32,
):
    if zernike_indices is None:
        zernike_indices = tuple(range(3, 16))
    optics_cfg = optics_cfg or {}
    pixel_size = float(optics_cfg.get('pixel_size', 0.065))
    wavelength = float(optics_cfg.get('lambda_emission', 0.525))
    na = float(optics_cfg.get('na', 1.0))
    coeffs = as_zernike_coeffs(coeffs, zernike_indices, optics_cfg)
    coeffs_t = torch.as_tensor(coeffs, device=device, dtype=dtype)
    if coeffs_t.ndim == 1:
        coeffs_t = coeffs_t.unsqueeze(0)
    rho, theta, pupil = pupil_coordinates_2d(
        image_size,
        pixel_size=pixel_size,
        wavelength=wavelength,
        na=na,
        device=coeffs_t.device,
        dtype=dtype,
    )
    wf = zernike_wavefront(zernike_indices, coeffs_t, rho, theta)
    return wf.squeeze(0).cpu().numpy(), pupil.cpu().numpy()


def coeffs_rms(a, b):
    a = np.asarray(a).ravel()
    b = np.asarray(b).ravel()
    return np.sqrt(np.mean((a - b) ** 2))


def _to_tensor(array):
    return torch.from_numpy(np.ascontiguousarray(np.asarray(array, dtype=np.float32)))[None]


def _first_present(mapping, *keys):
    for key in keys:
        if key in mapping and mapping[key] not in (None, ''):
            return mapping[key]
    return None


def _load_artifact(path, preferred_key=None):
    if path is None:
        return None
    path = Path(path)
    if not path.exists():
        return None
    suffix = path.suffix.lower()
    if suffix == '.npz':
        with np.load(path) as data:
            if preferred_key and preferred_key in data.files:
                return np.asarray(data[preferred_key])
            for key in ('psf', 'coefficients_um', 'coefficients', 'array'):
                if key in data.files:
                    return np.asarray(data[key])
            return np.asarray(data[data.files[0]])
    if suffix == '.npy':
        return np.load(path)
    if suffix in {'.txt', '.csv', '.tsv'}:
        delimiter = ',' if suffix == '.csv' else None
        return np.loadtxt(path, delimiter=delimiter)
    return load_image(path)


def _resolve_manifest_path(cfg, split='test', data_root_override=None, manifest_override=None):
    if manifest_override:
        return resolve_path(manifest_override, data_root_override)
    data_root = get_data_root(cfg, data_root_override)
    split_cfg = cfg.get('data', {}).get(split, {})
    manifest = split_cfg.get('manifest')
    if manifest:
        return resolve_path(manifest, data_root)
    image_dir = split_cfg.get('image_dir') or cfg.get('data', {}).get('val', {}).get('image_dir') or cfg.get('data', {}).get('train', {}).get('image_dir')
    if image_dir is None:
        raise KeyError(f'Cannot infer manifest path for split={split}')
    return resolve_path(image_dir, data_root).parent / 'metadata' / 'manifest.csv'


class ManifestInspectionDataset(Dataset):
    def __init__(self, manifest_path, data_root=None, normalize_percentile=(0.1, 99.9)) -> None:
        self.manifest_path = Path(manifest_path)
        self.manifest_root = infer_manifest_data_root(self.manifest_path, data_root)
        with self.manifest_path.open('r', newline='') as f:
            self.rows = list(csv.DictReader(f))
        if not self.rows:
            raise ValueError(f'No rows found in {self.manifest_path}')
        self.normalize_percentile = normalize_percentile

    def __len__(self) -> int:
        return len(self.rows)

    def _resolve(self, value):
        if value in (None, ''):
            return None
        return resolve_manifest_record_path(value, self.manifest_root)

    def _sample_optics(self, row):
        optics = dict(OPTICS_CFG or {})
        if row.get('pixel_size_um'):
            optics['pixel_size'] = float(row['pixel_size_um'])
        if row.get('na'):
            optics['na'] = float(row['na'])
        if row.get('lambda_em_um'):
            optics['lambda_emission'] = float(row['lambda_em_um'])
        if row.get('lambda_ex_um'):
            optics['lambda_excitation'] = float(row['lambda_ex_um'])
        if row.get('mode'):
            optics['mode'] = row['mode']
        return optics

    def __getitem__(self, index: int):
        row = self.rows[index % len(self.rows)]
        input_path = self._resolve(_first_present(row, 'abe_path', 'aberrated', 'input'))
        object_path = self._resolve(_first_present(row, 'object_path', 'obj_path', 'object'))
        target_path = self._resolve(_first_present(row, 'no_abe_path', 'target', 'gt'))
        psf_abe_path = self._resolve(_first_present(row, 'psf_abe_path', 'psf_path'))
        psf_no_abe_path = self._resolve(_first_present(row, 'psf_no_abe_path'))
        zernike_path = self._resolve(_first_present(row, 'zernike_path', 'coeff_path'))

        input_image = normalize01(load_image(input_path), self.normalize_percentile)
        sample = {
            'input': _to_tensor(input_image),
            'input_path': str(input_path),
            'object_path': str(object_path) if object_path else None,
            'target_path': str(target_path) if target_path else None,
            'psf_abe_path': str(psf_abe_path) if psf_abe_path else None,
            'psf_no_abe_path': str(psf_no_abe_path) if psf_no_abe_path else None,
            'zernike_path': str(zernike_path) if zernike_path else None,
            'pixel_size_um': float(row['pixel_size_um']) if row.get('pixel_size_um') else None,
            'na': float(row['na']) if row.get('na') else None,
            'lambda_em_um': float(row['lambda_em_um']) if row.get('lambda_em_um') else None,
            'lambda_ex_um': float(row['lambda_ex_um']) if row.get('lambda_ex_um') else None,
            'mode': row['mode'] if row.get('mode') else None,
            'rms_waves': float(row['rms_waves']) if row.get('rms_waves') else None,
            'optics_cfg': self._sample_optics(row),
            'row': row,
        }

        if object_path:
            sample['object'] = _to_tensor(normalize01(load_image(object_path), self.normalize_percentile))
        if target_path:
            sample['target'] = _to_tensor(normalize01(load_image(target_path), self.normalize_percentile))
        if psf_abe_path:
            sample['psf_abe'] = _to_tensor(_load_artifact(psf_abe_path, preferred_key='psf'))
        if psf_no_abe_path:
            sample['psf_no_abe'] = _to_tensor(_load_artifact(psf_no_abe_path, preferred_key='psf'))

        coeffs = None
        zernike_indices = None
        rms_waves = None
        if zernike_path:
            if Path(zernike_path).suffix.lower() == '.npz':
                with np.load(zernike_path) as data:
                    if 'coefficients_um' in data.files:
                        coeffs = np.asarray(data['coefficients_um'])
                    elif 'coefficients' in data.files:
                        coeffs = np.asarray(data['coefficients'])
                    if 'zernike_indices' in data.files:
                        zernike_indices = np.asarray(data['zernike_indices']).astype(int).tolist()
                    if 'rms_waves' in data.files:
                        rms_waves = float(np.asarray(data['rms_waves']).squeeze())
            else:
                coeffs = np.asarray(_load_artifact(zernike_path, preferred_key='coefficients_um'))
        if coeffs is not None:
            sample['true_coeffs'] = torch.as_tensor(np.asarray(coeffs, dtype=np.float32).squeeze())
        if zernike_indices is not None:
            sample['zernike_indices'] = torch.as_tensor(zernike_indices, dtype=torch.int64)
        if rms_waves is not None:
            sample['rms_waves'] = torch.tensor(rms_waves, dtype=torch.float32)

        return sample

repo: /root/dyx/AAAI
device: cuda


In [3]:
# 配置：根据需要修改路径/文件名
CONFIG_PATH = REPO_ROOT / 'configs' / 'self_supervised_2d.json'
CHECKPOINT_NAME = 'best.pt'
OUTPUT_DIR = None
DATA_ROOT_OVERRIDE = None  # 可改为你的数据根路径
MANIFEST_PATH_OVERRIDE = None  # 例如 /path/to/Testdata/metadata/manifest.csv
EVAL_SPLIT = 'test'
MAX_EVAL_SAMPLES = 8
N_ZERNIKE = None  # 将在加载 cfg 后自动读取
OPTICS_CFG = None  # 将在加载 cfg 后自动读取

In [2]:
# 加载模型与数据（按项目实际 API 调整）
try:
    with open(CONFIG_PATH, 'r') as f:
        cfg = json.load(f)
    OPTICS_CFG = cfg.get('optics', {})
    N_ZERNIKE = int(
        cfg.get('model', {}).get(
            'zernike_modes',
            len(cfg.get('model', {}).get('zernike_indices', OPTICS_CFG.get('zernike_indices', list(range(3, 16))))),
        )
    )
    output_dir = Path(cfg.get('output_dir', 'outputs/selfsupervised')) if OUTPUT_DIR is None else Path(OUTPUT_DIR)
    ckpt = output_dir / CHECKPOINT_NAME
    print('checkpoint:', ckpt, ckpt.exists())
    print('zernike_modes:', N_ZERNIKE)
    print('optics cfg:', OPTICS_CFG)
    model = make_model(cfg['model'])
    state = torch.load(ckpt, map_location='cpu') if ckpt.exists() else {}
    if 'model' in state:
        model.load_state_dict(state['model'], strict=False)
    model = model.to(device).eval()

    try:
        manifest_path = _resolve_manifest_path(cfg, split=EVAL_SPLIT, data_root_override=DATA_ROOT_OVERRIDE, manifest_override=MANIFEST_PATH_OVERRIDE)
        data_root = get_data_root(cfg, DATA_ROOT_OVERRIDE)
        print('manifest:', manifest_path)
        eval_set = ManifestInspectionDataset(
            manifest_path,
            data_root=data_root,
            normalize_percentile=tuple(cfg.get('data', {}).get('normalize_percentile', [0.1, 99.9])),
        )
        eval_loader = build_dataloader(eval_set, batch_size=1, shuffle=False, num_workers=0, drop_last=False)
        print('eval samples:', len(eval_set))
    except Exception as e:
        print('无法按 manifest 自动构建 eval_loader:', e)
        eval_loader = None
except Exception as e:
    print('加载模型或配置失败:', e)
    model = None
    eval_loader = None

加载模型或配置失败: name 'CONFIG_PATH' is not defined


## 2. 推理：获取网络输出与预测 Zernike 系数（示例流程）

In [ ]:
def first_present(mapping, *keys):
    for key in keys:
        if key in mapping and mapping[key] is not None:
            return mapping[key]
    return None


def to_cpu_or_none(value):
    if value is None:
        return None
    return value.detach().cpu() if torch.is_tensor(value) else np.asarray(value)


def batch_scalar(value):
    value = _batch_item(value)
    if torch.is_tensor(value) and value.ndim == 0:
        return value.item()
    if isinstance(value, (list, tuple)) and len(value) == 1:
        return value[0]
    return value


def sample_optics_from_batch(batch):
    optics = dict(OPTICS_CFG or {})
    mapping = {
        'pixel_size': batch_scalar(first_present(batch, 'pixel_size_um')),
        'na': batch_scalar(first_present(batch, 'na')),
        'lambda_emission': batch_scalar(first_present(batch, 'lambda_em_um')),
        'lambda_excitation': batch_scalar(first_present(batch, 'lambda_ex_um')),
        'mode': batch_scalar(first_present(batch, 'mode')),
    }
    for key, value in mapping.items():
        if value is not None:
            optics[key] = value
    return optics


examples = []
if eval_loader is None or model is None:
    print('请先确保 model 与 eval_loader 可用，然后重跑此单元。')
else:
    for idx, batch in enumerate(eval_loader):
        if idx >= MAX_EVAL_SAMPLES:
            break
        x = first_present(batch, 'input', 'aberrated', 'image')
        obj = first_present(batch, 'object', 'obj', 'target', 'gt')
        psf_abe = first_present(batch, 'psf_abe', 'psf', 'psf_abe_path')
        psf_no_abe = first_present(batch, 'psf_no_abe')
        true_coeffs = first_present(batch, 'true_coeffs', 'zernike', 'coeffs', 'target_coeffs')
        x = x.to(device) if torch.is_tensor(x) else torch.tensor(x).unsqueeze(0).to(device)
        with torch.no_grad():
            out = model(x)
        if isinstance(out, (list, tuple)) and len(out) == 2:
            recon, pred_coeffs = out
        else:
            recon = out if torch.is_tensor(out) else out[0]
            pred_coeffs = getattr(model, 'pred_coeffs', None)
            if pred_coeffs is None and isinstance(out, dict) and 'coeffs' in out:
                pred_coeffs = out['coeffs']
        examples.append({
            'x': x.cpu(),
            'recon': recon.cpu(),
            'obj': to_cpu_or_none(obj),
            'psf_abe': to_cpu_or_none(psf_abe),
            'psf_no_abe': to_cpu_or_none(psf_no_abe),
            'pred_coeffs': to_cpu_or_none(pred_coeffs),
            'true_coeffs': to_cpu_or_none(true_coeffs),
            'optics_cfg': sample_optics_from_batch(batch),
            'meta': {
                'idx': idx,
                'input_path': _batch_item(batch.get('input_path')),
                'object_path': _batch_item(batch.get('object_path')),
                'psf_abe_path': _batch_item(batch.get('psf_abe_path')),
                'psf_no_abe_path': _batch_item(batch.get('psf_no_abe_path')),
                'zernike_path': _batch_item(batch.get('zernike_path')),
                'rms_waves': _batch_item(batch.get('rms_waves')),
            },
        })
    print(f'evaluated {len(examples)} samples')

## 3. 可视化：网络输出、Zernike 相位图、真值与系数对比

In [ ]:
def to_np(t):
    if torch.is_tensor(t):
        return t.squeeze().detach().cpu().numpy()
    return np.asarray(t)


if not examples:
    print('没有示例可视化，请先运行推理单元。')
else:
    for ex in examples:
        recon = to_np(ex['recon'])
        obj = to_np(ex['obj']) if ex.get('obj') is not None else None
        psf_abe = to_np(ex['psf_abe']) if ex.get('psf_abe') is not None else None
        psf_no_abe = to_np(ex['psf_no_abe']) if ex.get('psf_no_abe') is not None else None
        sample_optics = ex.get('optics_cfg') or {}
        zernike_indices = sample_optics.get('zernike_indices')
        if zernike_indices is None and OPTICS_CFG:
            zernike_indices = OPTICS_CFG.get('zernike_indices', list(range(3, 16)))
        if zernike_indices is None:
            zernike_indices = list(range(3, 16))
        pred_coeffs = as_zernike_coeffs(ex['pred_coeffs'], zernike_indices, OPTICS_CFG)
        true_coeffs = as_zernike_coeffs(ex['true_coeffs'], zernike_indices, OPTICS_CFG)
        H = recon.shape[-2] if recon.ndim >= 2 else recon.shape[0]
        W = recon.shape[-1] if recon.ndim >= 2 else recon.shape[1] if recon.ndim >= 2 else recon.shape[0]
        image_size = (H, W)
        pred_phase, _ = coeffs_to_wavefront(
            pred_coeffs,
            image_size,
            zernike_indices=zernike_indices,
            optics_cfg=sample_optics or OPTICS_CFG,
        )
        true_phase, _ = coeffs_to_wavefront(
            true_coeffs,
            image_size,
            zernike_indices=zernike_indices,
            optics_cfg=sample_optics or OPTICS_CFG,
        )
        true_psf = psf_no_abe if psf_no_abe is not None else psf_abe

        coeffs_rms_val = coeffs_rms(pred_coeffs, true_coeffs)

        fig, axes = plt.subplots(2, 3, figsize=(16, 9), constrained_layout=True)
        axes = axes.ravel()

        axes[0].imshow(recon.squeeze(), cmap='gray')
        axes[0].set_title('network output (obj)')
        axes[0].axis('off')

        axes[1].imshow(pred_phase, cmap='RdBu')
        axes[1].set_title('predicted Zernike phase')
        axes[1].axis('off')

        if obj is not None:
            axes[2].imshow(obj.squeeze(), cmap='gray')
            axes[2].set_title('true obj')
        else:
            axes[2].text(0.5, 0.5, 'no true obj', ha='center')
        axes[2].axis('off')

        axes[3].imshow(true_phase, cmap='RdBu')
        axes[3].set_title('true Zernike phase')
        axes[3].axis('off')

        if true_psf is not None:
            axes[4].imshow(true_psf.squeeze(), cmap='magma')
            axes[4].set_title('manifest PSF')
        else:
            axes[4].text(0.5, 0.5, 'no PSF', ha='center')
        axes[4].axis('off')

        txt = (
            f"input: {ex['meta'].get('input_path')}\n"
            f"object: {ex['meta'].get('object_path')}\n"
            f"zernike: {ex['meta'].get('zernike_path')}\n"
            f"psf_abe: {ex['meta'].get('psf_abe_path')}\n"
            f"psf_no_abe: {ex['meta'].get('psf_no_abe_path')}\n\n"
            f"pred coeffs:\n{np.array2string(pred_coeffs, precision=3, separator=', ')}\n\n"
            f"true coeffs:\n{np.array2string(true_coeffs, precision=3, separator=', ')}\n\n"
            f"RMS coeffs: {coeffs_rms_val:.4f}"
        )
        axes[5].text(0.01, 0.5, txt, va='center', fontsize=8.5, family='monospace')
        axes[5].axis('off')
        plt.show()